## Bayes Optimal v Prompting for Combination Lock

In [1]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any

In [2]:
# styles = ["1", "2", "3", "4", "5", "6"]
reasoning = ["low", "medium", "high"]
styles = ["3", "4", "5", "6"]
results_paths: List[str] = [
    f'../src/optimal_explorer/strategies/combination_lock/logs/game_results/style{style}_{reasoning_effort}.jsonl'
    for style in styles for reasoning_effort in reasoning
]
bayes_optimal_path: str = f'../src/optimal_explorer/strategies/combination_lock/logs/bayes_optimal_2.jsonl'


models = [
    # "Gemini Pro 2.5",
    "DeepSeek R1",
    # "Claude Opus 4",
    # "Claude 3.5 Sonnet",
    # "OpenAI o3",
]

model_ids = [
    # "google/gemini-2.5-pro-preview",
    "deepseek/deepseek-r1-0528",
    # "anthropic/claude-opus-4",
    # "anthropic/claude-3.5-sonnet",
    # "openai/o3",
]

In [3]:
results = []

for reasoning_effort in reasoning:
    for style in styles:
        results_path = f'../src/optimal_explorer/strategies/combination_lock/logs/game_results/style{style}_{reasoning_effort}.jsonl'
        with open(results_path, 'r') as file:
            for line in file:
                data = json.loads(line)
                regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
                model = data['model']
                results.append({
                    'game_id': data['game_id'],
                    'model': model + f' (s={style}) ({reasoning_effort})',
                    'regret': regret,
                    'length': len(data['history']),
                    'style': style,
                    'reasoning_effort': reasoning_effort
                })
    results_df = pd.DataFrame(results)

bayes_optimal = []

with open(bayes_optimal_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
        bayes_optimal.append({
            'game_id': data['game_id'],
            'model': 'Bayes Optimal',
            'regret': regret,
            'length': len(data['history']),
            'style': style
        })
bayes_optimal_df = pd.DataFrame(bayes_optimal)

In [13]:
results_df.head(3)

,game_id,model,regret,length,style,reasoning_effort
0,89,deepseek/deepseek-r1-0528 (s=3) (low),"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",2,3,low
1,21,deepseek/deepseek-r1-0528 (s=3) (low),"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",4,3,low
2,32,deepseek/deepseek-r1-0528 (s=3) (low),"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",3,3,low


In [5]:
bayes_optimal_df.head(3)

,game_id,model,regret,length,style
0,0,Bayes Optimal,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",3,6
1,1,Bayes Optimal,"[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]",6,6
2,2,Bayes Optimal,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]",5,6


In [6]:
results_df = results_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
bayes_optimal_df = bayes_optimal_df.drop_duplicates(subset=['game_id', 'model'], keep='last')

In [7]:
len(results_df), len(bayes_optimal_df)

(1200, 100)

In [14]:
cumulative_regrets = []
error_bars = []

for reasoning_effort in reasoning:
    for style in styles:
        for mi, model in enumerate(models):
            model_df = results_df[results_df['model'] == model_ids[mi] + f' (s={style}) ({reasoning_effort})']
            # Convert regret lists to numpy array for easier computation
            regret_array = np.array(model_df['regret'].values.tolist())
            
            # Calculate mean regret per turn
            model_regret = np.mean(regret_array, axis=0)
            cumulative_regret = np.cumsum(model_regret)
            cumulative_regrets.append(cumulative_regret)
            
            # Calculate standard error of the mean for each turn
            sem = np.std(regret_array, axis=0) / np.sqrt(len(model_df))
            cumulative_sem = np.cumsum(sem)
            error_bars.append(cumulative_sem)
            
            print(f'{model} cumulative regret: {cumulative_regret}')

bayes_optimal_regret_array = np.array(bayes_optimal_df['regret'].values.tolist())
bayes_optimal_regret = np.mean(bayes_optimal_regret_array, axis=0)
bayes_optimal_cumulative_regret = np.cumsum(bayes_optimal_regret)
# Calculate standard error of the mean for each turn
sem = np.std(bayes_optimal_regret_array, axis=0) / np.sqrt(len(bayes_optimal_df))
bayes_optimal_cumulative_sem = np.cumsum(sem)

DeepSeek R1 cumulative regret: [1.   1.99 2.96 3.82 4.56 5.1  5.49 5.68 5.77 5.83 5.86 5.86]
DeepSeek R1 cumulative regret: [1.   1.99 2.96 3.83 4.56 5.11 5.47 5.7  5.86 5.96 6.01 6.01]
DeepSeek R1 cumulative regret: [1.   1.99 2.96 3.84 4.59 5.1  5.42 5.62 5.73 5.78 5.8  5.8 ]
DeepSeek R1 cumulative regret: [1.   1.99 2.96 3.86 4.59 5.16 5.54 5.79 5.96 6.04 6.09 6.12]
DeepSeek R1 cumulative regret: [1.   1.99 2.94 3.83 4.52 5.01 5.33 5.53 5.65 5.74 5.77 5.78]
DeepSeek R1 cumulative regret: [1.   1.98 2.96 3.88 4.7  5.26 5.66 5.91 6.02 6.06 6.06 6.06]
DeepSeek R1 cumulative regret: [1.   2.   2.97 3.87 4.6  5.14 5.46 5.7  5.82 5.88 5.91 5.91]
DeepSeek R1 cumulative regret: [1.   2.   2.96 3.84 4.56 5.1  5.47 5.67 5.79 5.83 5.84 5.84]
DeepSeek R1 cumulative regret: [1.   1.99 2.97 3.83 4.55 5.11 5.51 5.76 5.91 6.01 6.04 6.04]
DeepSeek R1 cumulative regret: [1.   2.   2.98 3.89 4.67 5.29 5.66 5.87 6.01 6.1  6.13 6.14]
DeepSeek R1 cumulative regret: [1.   1.99 2.97 3.89 4.64 5.14 5.5  5.7

In [15]:
cumulative_regrets

[array([1.  , 1.99, 2.96, 3.82, 4.56, 5.1 , 5.49, 5.68, 5.77, 5.83, 5.86,
        5.86]),
 array([1.  , 1.99, 2.96, 3.83, 4.56, 5.11, 5.47, 5.7 , 5.86, 5.96, 6.01,
        6.01]),
 array([1.  , 1.99, 2.96, 3.84, 4.59, 5.1 , 5.42, 5.62, 5.73, 5.78, 5.8 ,
        5.8 ]),
 array([1.  , 1.99, 2.96, 3.86, 4.59, 5.16, 5.54, 5.79, 5.96, 6.04, 6.09,
        6.12]),
 array([1.  , 1.99, 2.94, 3.83, 4.52, 5.01, 5.33, 5.53, 5.65, 5.74, 5.77,
        5.78]),
 array([1.  , 1.98, 2.96, 3.88, 4.7 , 5.26, 5.66, 5.91, 6.02, 6.06, 6.06,
        6.06]),
 array([1.  , 2.  , 2.97, 3.87, 4.6 , 5.14, 5.46, 5.7 , 5.82, 5.88, 5.91,
        5.91]),
 array([1.  , 2.  , 2.96, 3.84, 4.56, 5.1 , 5.47, 5.67, 5.79, 5.83, 5.84,
        5.84]),
 array([1.  , 1.99, 2.97, 3.83, 4.55, 5.11, 5.51, 5.76, 5.91, 6.01, 6.04,
        6.04]),
 array([1.  , 2.  , 2.98, 3.89, 4.67, 5.29, 5.66, 5.87, 6.01, 6.1 , 6.13,
        6.14]),
 array([1.  , 1.99, 2.97, 3.89, 4.64, 5.14, 5.5 , 5.75, 5.88, 5.95, 5.96,
        5.96]),
 array([1.

In [18]:
# Use Plotly's Dark24 color set for darker colors
colors = [px.colors.qualitative.Dark24[i] for i in [1, 10, 6, 15, 19]]

# Set LaTeX font for all text elements
latex_font = dict(
    family="Latin Modern Roman, Times New Roman, serif",
    size=14,
    color="black"
)

fig = go.Figure()

for ri, reasoning_effort in enumerate(reasoning):
    for si, style in enumerate(styles):
        for mi, model in enumerate(models):
            x_vals = list(range(1, 13))
            y_mean = cumulative_regrets[mi + si]
            y_err = error_bars[mi + si]
            color = colors[(mi + si) % len(colors)]

            # Add shaded error region (as a filled area)
            fig.add_trace(go.Scatter(
                x=x_vals + x_vals[::-1],
                y=(y_mean + y_err).tolist() + (y_mean - y_err)[::-1].tolist(),
                fill='toself',
                fillcolor=f'rgba{tuple(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)) + (0.18,)}',
                line=dict(color='rgba(255,255,255,0)'),
                hoverinfo="skip",
                showlegend=False,
                name=f"{model} (s={style}) (r={reasoning_effort})"
            ))

            # Add main mean curve, thicker
            fig.add_trace(go.Scatter(
                x=x_vals,
                y=y_mean,
                mode='lines+markers',
                name=model + f' (s={style}) (r={reasoning_effort})',
                line=dict(width=4, color=color),
                marker=dict(size=6, color=color)
            ))

# Add Bayes Optimal shaded error region (as a filled area)
fig.add_trace(go.Scatter(
    x=list(range(1, 13)) + list(range(1, 13))[::-1],
    y=(bayes_optimal_cumulative_regret + bayes_optimal_cumulative_sem).tolist() + (bayes_optimal_cumulative_regret - bayes_optimal_cumulative_sem)[::-1].tolist(),
    fill='toself',
    fillcolor=f'rgba(0, 0, 0, 0.5)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False,
    name='Bayes Optimal'
))
# Add Bayes Optimal line
fig.add_trace(go.Scatter(
    x=list(range(1, 13)),
    y=bayes_optimal_cumulative_regret,
    mode='markers+lines+lines',
    name='Bayes Optimal',
    line=dict(width=4, color='rgba(0, 0, 0, 0.8)'),
    marker=dict(size=6, color='rgba(0, 0, 0, 0.8)')
))

# Add y=x baseline as a dashed line
baseline_x = list(range(1, 13))
baseline_y = list(range(1, 13))
fig.add_trace(go.Scatter(
    x=baseline_x,
    y=baseline_y,
    mode='lines',
    name='Baseline',
    line=dict(color='black', width=2, dash='dash'),
    showlegend=True
))

fig.update_layout(
    width=600,
    height=470,
    title=dict(
        text='',
        font=latex_font
    ),
    xaxis_title="Episode (Symbol Combo-Lock)",
    yaxis_title="Cumulative Regret",
    font=latex_font,
    xaxis=dict(
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis=dict(
        range=[1, 12],
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        side='right',  # default, but we want ticks on both sides
        showticksuffix='all',
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis2=dict(
        overlaying='y',
        side='right',
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    legend=dict(
        title='',
        x=0.03,  # left edge, inside plot
        y=0.97,  # top edge, inside plot
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='black',
        borderwidth=1,
        font=latex_font
    ),
    template='plotly_white'
)

# Add yaxis2 to all traces so ticks show on both sides
for trace in fig.data:
    trace.update(yaxis='y')

In [19]:
fig.show()

In [50]:
# TODOs

# Checking new bayes-optimal strategy [done]
# plot output and input token usage in separate graphs [done]
# 